In [4]:
import pandas as pd
import os

arquivo_excel = 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'

# leitura das abas
df_2022 = pd.read_excel(arquivo_excel, sheet_name='PEDE2022')
df_2023 = pd.read_excel(arquivo_excel, sheet_name='PEDE2023')
df_2024 = pd.read_excel(arquivo_excel, sheet_name='PEDE2024')

# setorização dos anos do database
df_2022['Ano'] = 2022
df_2023['Ano'] = 2023
df_2024['Ano'] = 2024

# renomeação das colunas
df_2022.rename(columns={
    'Nome': 'Nome',
    'Ano nasc': 'Data de Nasc',
    'Idade 22': 'Idade',
    'Pedra 22': 'Pedra',
    'INDE 22': 'INDE',
    'Matem': 'Mat',
    'Portug': 'Por',
    'Inglês': 'Ing',
    'Fase ideal': 'Fase Ideal',
    'Defas': 'Defasagem'
}, inplace=True)

df_2023.rename(columns={
    'Nome Anonimizado': 'Nome',
    'Pedra 2023': 'Pedra',
    'INDE 2023': 'INDE'
}, inplace=True)

df_2024.rename(columns={
    'Nome Anonimizado': 'Nome',
    'Pedra 2024': 'Pedra',
    'INDE 2024': 'INDE'
}, inplace=True)

# remove colunas
colunas_para_remover = ['Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'INDE 22', 'INDE 23']

for df in [df_2022, df_2023, df_2024]:
    df.drop(columns=[c for c in colunas_para_remover if c in df.columns], inplace=True)

# concatenar data base
df = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

# limpa textos indesejados
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# preenche vazios
df.replace(r'^\s*$', pd.NA, regex=True, inplace=True)

# corrigi datas
import pandas as pd
import numpy as np

def tratar_data(valor):
    try:
        if pd.isna(valor):
            return pd.NaT

        valor_str = str(valor).strip()

        # se for só ano (ex: 2004)
        if valor_str.isdigit() and len(valor_str) == 4:
            return pd.to_datetime(valor_str + '-01-01')

        return pd.to_datetime(valor_str, errors='coerce')

    except:
        return pd.NaT

# aplicar
df['Data de Nasc'] = df['Data de Nasc'].apply(tratar_data)

# substituir a própria coluna pelo ano
df['Data de Nasc'] = df['Data de Nasc'].dt.year

# corrige idade
def corrigir_idade(valor):
    try:
        if pd.isna(valor):
            return None

        # caso venha como data tipo 1900-01-11
        if isinstance(valor, pd.Timestamp):
            return valor.day

        # caso venha como número serial do Excel
        if isinstance(valor, (int, float)) and valor > 100:
            # converte serial Excel → data → pega dia
            data = pd.to_datetime('1899-12-30') + pd.to_timedelta(valor, 'D')
            return data.day

        # caso string tipo data
        if isinstance(valor, str) and '1900' in valor:
            data = pd.to_datetime(valor, errors='coerce')
            return data.day

        return int(valor)

    except:
        return None

if 'Idade' in df.columns:
    df['Idade'] = df['Idade'].apply(corrigir_idade)

# corrige gênero
if 'Gênero' in df.columns:

    df['Gênero'] = df['Gênero'].astype(str).str.strip().str.lower()

    df['Gênero'] = df['Gênero'].replace({
        'menina': 'Feminino',
        'feminino': 'Feminino',
        'menino': 'Masculino',
        'masculino': 'Masculino'
    })

    df['Gênero'] = df['Gênero'].str.title()

# salva arquivo para aplicar ml
nome_arquivo = 'BASE_FINAL_TRATADA.xlsx'
df.to_excel(nome_arquivo, index=False)

try:
    os.startfile(nome_arquivo)
except:
    pass

C:\Users\isabe\AppData\Local\Temp\ipykernel_25636\1455687188.py:52: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
